# Decision Tree Implementation
### Hands-on Notebook — Building & Visualizing Decision Trees

**Learning Objectives**
1. Build Decision Trees for **category prediction**
2. Visualize **tree-based decision logic**

**Subtopics:** Decision Trees · Splits & Gini Impurity · Tree Visualization

---
**The situation:** same Pass/Fail question as last session, but now with two features — hours practiced and whether the learner attended live doubt sessions. Instead of a single smooth probability curve, we'll let the model learn a set of nested yes/no rules.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

plt.rcParams['figure.figsize'] = (7, 4.5)
np.random.seed(42)


## 1. The dataset

The exact 8-learner worked example from the slides.


In [ ]:
demo = pd.DataFrame({
    "hours":    [2, 3, 2, 3, 5, 6, 5, 6],
    "attended": [0, 0, 1, 1, 0, 0, 1, 1],   # 1 = attended live sessions, 0 = did not
    "passed":   [0, 0, 0, 1, 1, 1, 1, 1],
})
demo


## 2. Gini impurity — from scratch

$$Gini = 1 - \sum_i p_i^2$$

Let's compute impurity for the root node (all 8 learners), matching the slide.


In [ ]:
def gini(labels):
    labels = np.array(labels)
    n = len(labels)
    if n == 0:
        return 0
    impurity = 1.0
    for cls in np.unique(labels):
        p = np.sum(labels == cls) / n
        impurity -= p ** 2
    return impurity

root_gini = gini(demo["passed"])
print(f"Root node Gini: {root_gini:.3f}")   # should be 0.5


### Testing a candidate split: `hours >= 4`

We'll compute the weighted Gini after this split, exactly like the slide's worked example.


In [ ]:
def weighted_gini_after_split(df, condition):
    left = df[~condition]
    right = df[condition]
    n = len(df)
    g_left = gini(left["passed"])
    g_right = gini(right["passed"])
    weighted = (len(left) / n) * g_left + (len(right) / n) * g_right
    return g_left, g_right, weighted

condition = demo["hours"] >= 4
g_left, g_right, weighted = weighted_gini_after_split(demo, condition)

print(f"Left (hours < 4)  Gini: {g_left:.3f}   n={sum(~condition)}")
print(f"Right (hours >= 4) Gini: {g_right:.3f}   n={sum(condition)}")
print(f"Weighted Gini after split: {weighted:.3f}")
print(f"Impurity reduction: {root_gini:.3f} - {weighted:.3f} = {root_gini - weighted:.3f}")


**Checkpoint:** this should match the slide — weighted Gini ≈ 0.1875, a solid drop from the root's 0.5. That's why `hours >= 4` wins as the root split.


### Compare against another candidate: `attended == 1`

Let's confirm the tree wouldn't have preferred this split instead.


In [ ]:
condition2 = demo["attended"] == 1
g_left2, g_right2, weighted2 = weighted_gini_after_split(demo, condition2)
print(f"Weighted Gini after splitting on 'attended': {weighted2:.3f}")
print(f"Weighted Gini after splitting on 'hours >= 4': {weighted:.3f}")
print("Lower weighted Gini wins — hours >= 4 is the better root split." if weighted < weighted2 else "attended wins")


## 3. Fitting a real Decision Tree with scikit-learn

Now let's let scikit-learn find the best splits automatically, and confirm it agrees with our by-hand root split.


In [ ]:
X = demo[["hours", "attended"]]
y = demo["passed"]

clf = DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=42)
clf.fit(X, y)

train_acc = accuracy_score(y, clf.predict(X))
print(f"Training accuracy: {train_acc:.2f}")


## 4. Visualizing the tree — flowchart view

`plot_tree()` draws exactly the kind of diagram from the slides: root, decision nodes, and leaves, with Gini and class counts at every node.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_tree(
    clf,
    feature_names=["hours", "attended"],
    class_names=["Fail", "Pass"],
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax
)
plt.title("Decision Tree — Pass/Fail")
plt.show()


## 5. Visualizing the tree — decision regions

This is the other half of "visualizing tree-based logic": instead of the flowchart, we plot the actual rectangular regions the tree carves out of feature space.


In [ ]:
def plot_decision_regions(model, X, y, feature_names):
    x_min, x_max = X.iloc[:, 0].min() - 1, X.iloc[:, 0].max() + 1
    y_min, y_max = X.iloc[:, 1].min() - 0.5, X.iloc[:, 1].max() + 0.5

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    fig, ax = plt.subplots()
    ax.contourf(xx, yy, Z, alpha=0.35, cmap="RdYlGn", levels=[-0.5, 0.5, 1.5])
    scatter = ax.scatter(X.iloc[:, 0], X.iloc[:, 1], c=y, cmap="RdYlGn",
                          edgecolor="#122238", s=90, zorder=3)
    ax.set_xlabel(feature_names[0])
    ax.set_ylabel(feature_names[1])
    ax.set_title("Decision regions: rectangular, axis-aligned splits")
    plt.tight_layout()
    plt.show()

plot_decision_regions(clf, X, y, ["hours", "attended"])


Notice the straight, axis-aligned edges — every boundary is a cut along one feature at a time. Compare this mentally to last session's smooth sigmoid boundary.


## 6. Making a prediction — tracing a new learner

A new learner practiced 3 hours and did **not** attend live sessions. Let's trace it through the tree, then confirm with `.predict()`.


In [ ]:
new_learner = pd.DataFrame({"hours": [3], "attended": [0]})
prediction = clf.predict(new_learner)[0]
probabilities = clf.predict_proba(new_learner)[0]

print("Predicted class:", "Pass" if prediction == 1 else "Fail")
print("Class probabilities [Fail, Pass]:", probabilities)


## 7. Depth & overfitting

Let's see the underfit → good fit → overfit progression by varying `max_depth` on a slightly larger synthetic dataset.


In [ ]:
n_samples = 60
hours_full = np.random.uniform(0, 8, n_samples)
attended_full = np.random.randint(0, 2, n_samples)
# true rule with some noise
passed_full = ((hours_full >= 4) | ((hours_full >= 2.5) & (attended_full == 1))).astype(int)
flip = np.random.rand(n_samples) < 0.08
passed_full = np.where(flip, 1 - passed_full, passed_full)

df_full = pd.DataFrame({"hours": hours_full, "attended": attended_full, "passed": passed_full})
X_full, y_full = df_full[["hours", "attended"]], df_full["passed"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, depth in zip(axes, [1, 3, None]):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_full, y_full)
    acc = accuracy_score(y_full, model.predict(X_full))

    x_min, x_max = X_full["hours"].min() - 1, X_full["hours"].max() + 1
    y_min, y_max = -0.5, 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.35, cmap="RdYlGn", levels=[-0.5, 0.5, 1.5])
    ax.scatter(X_full["hours"], X_full["attended"], c=y_full, cmap="RdYlGn", edgecolor="#122238", s=40)
    label = "None (full depth)" if depth is None else depth
    ax.set_title(f"max_depth={label}\ntrain accuracy={acc:.2f}")
    ax.set_xlabel("hours")
    ax.set_ylabel("attended")

plt.tight_layout()
plt.show()


Notice: `max_depth=1` misses real structure (underfits), `max_depth=None` chases every noisy point with tiny regions (overfits — high train accuracy, but likely to generalize poorly), and `max_depth=3` sits in between. In practice you'd confirm the right depth using a validation/test set, not training accuracy alone.


## 8. Exercises

Work through these before checking the solutions.


### Exercise 1 — Compute Gini by hand
For a node with 6 Pass and 2 Fail, compute the Gini impurity by hand, then verify with the `gini()` function above.


In [ ]:
# TODO: labels = [1]*6 + [0]*2, compute manually, then check with gini()



<details><summary>Show solution</summary>

```python
p_pass, p_fail = 6/8, 2/8
manual = 1 - (p_pass**2 + p_fail**2)
print(manual)                       # 0.375
print(gini([1]*6 + [0]*2))          # matches
```
</details>


### Exercise 2 — Try a different candidate split
On the original 8-row `demo` dataset, test the split `hours >= 5` instead of `hours >= 4`. Is its weighted Gini better or worse than the split we chose? Why might the tree still prefer `hours >= 4`?


In [ ]:
# TODO: use weighted_gini_after_split(demo, demo['hours'] >= 5)



<details><summary>Show solution</summary>

```python
g_left3, g_right3, weighted3 = weighted_gini_after_split(demo, demo["hours"] >= 5)
print(weighted3)
```
`hours >= 5` gives a *higher* weighted Gini than `hours >= 4` (it separates the classes less cleanly), so the tree correctly prefers `hours >= 4` — it isn't just picking any threshold, it's searching for the one that reduces impurity the most.
</details>


### Exercise 3 — Read the tree diagram
Using the `plot_tree()` output from Section 4, find the leaf with the worst (highest) Gini value. How many training samples fall into it, and what does that tell you about the limits of this tiny dataset?


<details><summary>Show solution</summary>

The `attended == 1` leaf under the `hours < 4` branch has Gini = 0.5 with only 2 samples (1 Pass, 1 Fail) — the tree has no way to separate them further with the features it has. This mirrors the slide's point: small or insufficiently informative data leaves some impurity that no split can remove.
</details>


## 9. Recap

- A Decision Tree **builds category predictions** by recursively splitting on the feature/threshold that reduces **Gini impurity** the most, until leaves are pure or a stopping rule (like `max_depth`) is hit.
- **Visualizing tree logic** works two ways: the flowchart (`plot_tree()`) shows the sequence of questions; the decision-region plot shows the resulting rectangular, axis-aligned regions in feature space.
- Tree depth is a trade-off — too shallow underfits, too deep overfits by memorizing noise.

**Next up:** evaluating classifiers more rigorously (train/test splits, cross-validation) and ensemble methods that combine many trees.
